In [1]:
import os
import sys
import torch
import yaml
import matplotlib.pyplot as plt


# --- Setup paths to import custom modules ---
# Go up two levels from the current script's directory (plot_scripts -> code -> project_root)
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
code_root = os.path.join(project_root, 'src')
if code_root not in sys.path:
    sys.path.insert(0, code_root)

from utilities import STFTtransform
from datasets_local import PrecomputedDataset

# --- Add this section to teach PyYAML about the custom tags ---
def tuple_constructor(loader: yaml.SafeLoader, node: yaml.nodes.Node) -> tuple:
    """Converts a YAML sequence tagged with !tuple or !range into a Python tuple."""
    return tuple(loader.construct_sequence(node))

yaml.add_constructor("!tuple", tuple_constructor, Loader=yaml.SafeLoader)
yaml.add_constructor("!range", tuple_constructor, Loader=yaml.SafeLoader)
# --- End of section ---


# --- Load Configuration from YAML files ---
config = {}
config_base_path = os.path.join(project_root, "src", "configs")
# with open(os.path.join(config_base_path, "datasets/brudex.yaml"), "r") as f:
with open(os.path.join(config_base_path, "datasets/J1_dataset_deactivation.yaml"), "r") as f:
    config["dataset"] = yaml.safe_load(f)
with open(os.path.join(config_base_path, "algorithms/cosad_ICASSP_4_plots.yaml"), "r") as f:
    config["cosad_algorithm"] = yaml.safe_load(f)

print("--- Configuration Loaded ---")
print(f"Dataset ID: {config['dataset']['id']}")
# print(f"Precompute Mode: {config['dataset']['precompute_mode']}")
# assert config['dataset']['precompute_mode'] == 'features', "This notebook requires precompute_mode to be 'features'."

# --- Extract Parameters from Config ---
# STFT and feature parameters
algo_params = config["cosad_algorithm"]
fs = algo_params["sampling_frequency"]
frame_length_s = algo_params["frame_length"]
frame_shift_s = algo_params["frame_shift"]
window_type = algo_params["window_type"]
feature_extractor = algo_params["cosad_features"][0]["WGMSC_Feature_Extractor"]
stc = feature_extractor["smoothing_time_constant"]
wtc = feature_extractor["whitening_time_constant"]
stcr = feature_extractor["smoothing_time_constant_rev"]
wtcr = feature_extractor["whitening_time_constant_rev"]
# Instantiate the transform to get nfft and hop_length
transform = STFTtransform(
    frame_length=frame_length_s,
    frame_shift=frame_shift_s,
    sampling_frequency=fs,
    window_type=window_type,
)
nfft = transform.nfft
hop_length = transform.hop_length

# Construct the unique ID for the feature configuration, matching BrudexDataModule
feature_config_id = (
    f"WGMSC_nb_fl{frame_length_s}_fs{frame_shift_s}_sf{fs}_win{window_type}"
    f"_stc{stc}_wtc{wtc}_stcr{stc}_wtcr{wtc}"
)
print(f"Feature Config ID: {feature_config_id}")

# --- Load the Dataset ---
# Construct the path to the pre-computed test features
precomputed_base_path = os.path.join(project_root, "databases", "precomputed")
feature_path = os.path.join(precomputed_base_path, config['dataset']['id'], "features", feature_config_id, "test")

print(f"\nLoading test dataset from: {feature_path}")

# Instantiate the dataset for the test split
# preload_to_ram=True is fast for validation/testing if memory allows
test_dataset = PrecomputedDataset(precomputed_dir=feature_path, preload_to_ram=True)

print(f"Successfully loaded {len(test_dataset)} test scenarios.")

--- Configuration Loaded ---
Dataset ID: J1_BXLS_deactivation
Feature Config ID: WGMSC_nb_fl0.064_fs0.016_sf16000_winsqrt-hann_stc0.5_wtc0.5_stcr0.5_wtcr0.5

Loading test dataset from: /data4/Henri/j3/framewiseSpeakerCounting/databases/precomputed/J1_BXLS_deactivation/features/WGMSC_nb_fl0.064_fs0.016_sf16000_winsqrt-hann_stc0.5_wtc0.5_stcr0.5_wtcr0.5/test
Initialized dataset with 750 files from 1 directories.
Pre-loading 750 files into RAM...


Pre-loading data: 100%|██████████| 750/750 [02:43<00:00,  4.58it/s]

...pre-loading complete.
Successfully loaded 750 test scenarios.


In [2]:
feature_extractor

{'smoothing_time_constant': 0.5,
 'whitening_time_constant': 0.5,
 'smoothing_time_constant_rev': 0.5,
 'whitening_time_constant_rev': 0.5}

In [3]:
algo_params

{'sampling_frequency': 16000,
 'frame_length': 0.064,
 'frame_shift': 0.016,
 'window_type': 'sqrt-hann',
 'smoothing_time_constant': 1.0,
 'whitening_time_constant': 1.0,
 'cosad_features': [{'WGMSC_Feature_Extractor': {'smoothing_time_constant': 0.5,
    'whitening_time_constant': 0.5,
    'smoothing_time_constant_rev': 0.5,
    'whitening_time_constant_rev': 0.5}},
  {'GMSC_Feature_Extractor': {'smoothing_time_constant': 0.5}},
  {'STFT_Feature_Encoder': {}}],
 'cosad_estimators': [{'COSAD_Threshold_Detector': {'smoothing_time_constant': 0.5,
    'coherence_threshold': 0.35,
    'min_activation_time_difference': 0.5,
    'smoothing_time_constant_rev': 0.5,
    'coherence_threshold_rev': 0.67,
    'min_deactivation_time_difference': 0.5,
    'detect_deactivations': True}},
  {'COSAD_RNN_Detector': {'hidden_size': 0.5,
    'num_layers': 3,
    'dropout': 0.0,
    'bias': True,
    'use_rev_features': True}},
  {'COSAD_TCNofficial_Detector': {'BN_dim': 128,
    'hidden_dim': 256,
    '

In [4]:
import numpy as np
from utilities.math4torch import exp_windowing
import matplotlib as mpl

# --- LaTeX/PGF Setup for Paper-Quality Figures ---
# This block ensures the plot uses the same fonts as your LaTeX document.
mpl.use("pgf")
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'serif',
    'text.usetex': True,
    'pgf.rcfonts': False,
    'pgf.preamble': r'\usepackage{amsmath,graphicx,siunitx,bm}' # Added bm for \bm command
})

# --- Load a Single Scenario ---
# Choose which scenario to inspect by changing the index
scenario_idx = 45
if scenario_idx >= len(test_dataset):
    raise IndexError(f"scenario_idx {scenario_idx} is out of bounds for dataset with {len(test_dataset)} samples.")

# Get the data for the chosen scenario
scenario_data = test_dataset[scenario_idx]
print(scenario_data.keys())
features = scenario_data['features']
metadata = scenario_data['meta']

print(f"\n--- Inspecting Scenario {scenario_idx} (ID: {metadata.get('scenario_id', 'N/A')}) ---")

# --- Get Smoothing Parameters from Config ---
threshold_params = config["cosad_algorithm"]["cosad_estimators"][0]["COSAD_Threshold_Detector"]
smoothing_tau_act = threshold_params["smoothing_time_constant"]
smoothing_tau_deact = threshold_params["smoothing_time_constant_rev"]

# --- Calculate Smoothing Factors (beta) ---
smoothing_factor_act = transform.timeConstant2smoothingFactor(smoothing_tau_act)
smoothing_factor_deact = transform.timeConstant2smoothingFactor(smoothing_tau_deact)

num_freqs = features.shape[0] // 2

# --- Apply Smoothing using the imported exp_windowing function ---
smoothed_wgmsc_act_wide_tensor = exp_windowing(
    features[:num_freqs], smoothing_factor=smoothing_factor_act, dim=-1
)
smoothed_wgmsc_deact_wide_tensor = exp_windowing(
    features[num_freqs:], smoothing_factor=smoothing_factor_deact, dim=-1
)

# --- Extract and Convert Data for Plotting ---
wgmsc_act_narrow = features[:num_freqs].cpu().numpy()
wgmsc_deact_narrow = features[num_freqs:].cpu().numpy()
wgmsc_act_wide = features['wgmsc_wideband'].cpu().numpy()
wgmsc_deact_wide = features['wgmsc_wideband_rev'].cpu().numpy()
smoothed_wgmsc_act_wide = smoothed_wgmsc_act_wide_tensor.cpu().numpy()
smoothed_wgmsc_deact_wide = smoothed_wgmsc_deact_wide_tensor.cpu().numpy()
gt_source_count = metadata['source_count'].cpu().numpy()

# --- Create Time and Frequency Axes ---
# The last dimension (shape[-1]) is the time/frame dimension.
num_frames = wgmsc_act_narrow.shape[-1]
time_axis = np.arange(num_frames) * frame_shift_s
freq_axis = transform.frequencies().cpu().numpy()

# --- Figure Size for Paper ---
# 1 inch = 72 points. LaTeX column width is 244.6937 pt.
columnwidth_in = 506.45905 / 72
fig_width = columnwidth_in
# Adjust height for aspect ratio and readability of 3 subplots.
# A value around 4.5 might be a good starting point.
fig_height = 3.8

# --- Font Size Settings ---
# Adjust these values to scale the fonts in the plot
suptitle_fontsize = 20
title_fontsize = 10
label_fontsize = 9
tick_fontsize = 9
legend_fontsize = 9

# --- Create the Consolidated Plot ---
fig, axs = plt.subplots(3, 1, figsize=(fig_width, fig_height), sharex=True, gridspec_kw={'height_ratios': [1, 1, 1]})
# fig.suptitle(f"Scenario Analysis: {metadata.get('scenario_id', 'N/A')}", fontsize=suptitle_fontsize)

# --- Determine shared color scale for the two image plots ---
vmin = min(wgmsc_act_narrow.min(), wgmsc_deact_narrow.min())
vmax = max(wgmsc_act_narrow.max(), wgmsc_deact_narrow.max())

# --- 1. Narrowband Activation Features ---
im1 = axs[0].imshow(wgmsc_act_narrow, aspect='auto', origin='lower',
                    extent=[time_axis[0], time_axis[-1], freq_axis[0], freq_axis[-1]],
                    cmap='inferno', vmin=vmin, vmax=vmax)
axs[0].set_title(r"Narrowband GMSC Features $\bm{\gamma}^{\text{a}}_{t}$ (Activation)", fontsize=title_fontsize)
axs[0].set_ylabel("Frequency (Hz)", fontsize=label_fontsize)
axs[0].tick_params(axis='y', labelsize=tick_fontsize)


# Color for number of sources overlay
num_sources_color = 'cyan'
num_sources_color_label = 'black' # Match label color to line color for clarity

# Add source count overlay to the first plot
ax0_twin = axs[0].twinx()
ax0_twin.step(time_axis, gt_source_count, where='post', color=num_sources_color, linewidth=1.5, alpha=0.8)
ax0_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)
ax0_twin.tick_params(axis='y', labelcolor=num_sources_color_label, labelsize=tick_fontsize)
ax0_twin.set_ylim(-0.1, gt_source_count.max() + 1.1)
ax0_twin.set_yticks(np.arange(0, gt_source_count.max() + 1, 1))
ax0_twin.grid(False)


# --- 2. Narrowband Deactivation Features ---
im2 = axs[1].imshow(wgmsc_deact_narrow, aspect='auto', origin='lower',
                    extent=[time_axis[0], time_axis[-1], freq_axis[0], freq_axis[-1]],
                    cmap='inferno', vmin=vmin, vmax=vmax)
axs[1].set_title(r"Narrowband GMSC Features $\bm{\gamma}^{\text{d}}_{t}$ (Deactivation)", fontsize=title_fontsize)
axs[1].set_ylabel("Frequency (Hz)", fontsize=label_fontsize)
axs[1].tick_params(axis='y', labelsize=tick_fontsize)

# Add source count overlay to the second plot
ax1_twin = axs[1].twinx()
ax1_twin.step(time_axis, gt_source_count, where='post', color=num_sources_color, linewidth=1.5, alpha=0.8)
ax1_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)
ax1_twin.tick_params(axis='y', labelcolor=num_sources_color_label, labelsize=tick_fontsize)
ax1_twin.set_ylim(-0.1, gt_source_count.max() + 1.1)
ax1_twin.set_yticks(np.arange(0, gt_source_count.max() + 1, 1))
ax1_twin.grid(False)

# --- Add a single shared vertical colorbar for the whole figure ---
cbar = fig.colorbar(im1, ax=axs, label='GMSC Value', orientation='vertical', shrink=0.66, anchor=(2.3, 1.2))
cbar.set_label('GMSC Value', size=label_fontsize)
cbar.ax.tick_params(labelsize=tick_fontsize)




# Plot ground truth source count on a secondary y-axis
ax3_twin = axs[2].twinx()
ax3_twin.step(time_axis, gt_source_count, where='post', label='Ground-Truth Source Count', color=num_sources_color, linewidth=2)
ax3_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)
ax3_twin.tick_params(axis='y', labelcolor=num_sources_color_label, labelsize=tick_fontsize)
ax3_twin.set_ylim(-0.1, gt_source_count.max() + 1.1)
ax3_twin.set_yticks(np.arange(0, gt_source_count.max() + 1, 1))
ax3_twin.grid(False)

# --- 3. Smoothed Broadband Features and Ground Truth ---
# Plot raw features for comparison
# axs[2].plot(time_axis, wgmsc_act_wide, color='lightblue', linestyle=':', label='Raw GMSC (Act)')
# axs[2].plot(time_axis, wgmsc_deact_wide, color='moccasin', linestyle=':', label='Raw GMSC (Deact)')
# Plot smoothed features
axs[2].plot(time_axis, smoothed_wgmsc_act_wide, label=r"$\bar{\gamma}_{t}^{\text{a}}$ (Act)", color='darkred')
axs[2].plot(time_axis, smoothed_wgmsc_deact_wide, label=r"$\bar{\gamma}_{t}^{\text{d}}$ (Deact)", color='darkorange', linestyle='--')

# Combine legends from both axes
lines, labels = axs[2].get_legend_handles_labels()
lines2, labels2 = ax3_twin.get_legend_handles_labels()
axs[2].legend(lines + lines2, labels + labels2, loc='upper left', ncol=3, fontsize=legend_fontsize)

axs[2].set_title("Smoothed Broadband GMSC Features", fontsize=title_fontsize)
axs[2].set_xlabel("Time (s)", fontsize=label_fontsize)
axs[2].set_ylabel("GMSC Value", fontsize=label_fontsize)
axs[2].tick_params(axis='both', labelsize=tick_fontsize)
axs[2].set_ylim(0, 1)
axs[2].grid(True)

# --- Final Touches ---
plt.tight_layout(pad=0.1)
# Instead of plt.show(), save the figure to a file
plt.savefig("feature_figure_test.pdf", bbox_inches='tight', dpi=600)
# plt.savefig("/data2/Henri/Journal3/framewiseSpeakerCounting/documents/ICASSP2026/figures/feature_figure.pdf", bbox_inches='tight', dpi=600)

# To see the plot in the notebook, you might need to switch back to the default backend
# mpl.use("module://matplotlib_inline.backend_inline")
# plt.show()

<>:104: SyntaxWarning: invalid escape sequence '\#'
<>:122: SyntaxWarning: invalid escape sequence '\#'
<>:139: SyntaxWarning: invalid escape sequence '\#'
<>:104: SyntaxWarning: invalid escape sequence '\#'
<>:122: SyntaxWarning: invalid escape sequence '\#'
<>:139: SyntaxWarning: invalid escape sequence '\#'
/tmp/ipykernel_1692117/1368201446.py:104: SyntaxWarning: invalid escape sequence '\#'
  ax0_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)
/tmp/ipykernel_1692117/1368201446.py:122: SyntaxWarning: invalid escape sequence '\#'
  ax1_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)
/tmp/ipykernel_1692117/1368201446.py:139: SyntaxWarning: invalid escape sequence '\#'
  ax3_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)
/tmp/ipykernel_1692117/1368201446.py:52: UserWarning: Using a non-tuple sequence for multidimensional indexing is deprecated and will be changed in pytorc

dict_keys(['features', 'input_type', 'feature_info', 'meta'])

--- Inspecting Scenario 45 (ID: J1_BXLS_deactivation_test_generator_139) ---


/tmp/ipykernel_1692117/1368201446.py:104: SyntaxWarning: invalid escape sequence '\#'
  ax0_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)
/tmp/ipykernel_1692117/1368201446.py:122: SyntaxWarning: invalid escape sequence '\#'
  ax1_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)
/tmp/ipykernel_1692117/1368201446.py:139: SyntaxWarning: invalid escape sequence '\#'
  ax3_twin.set_ylabel("\# Sources", color=num_sources_color_label, fontsize=label_fontsize)


IndexError: too many indices for tensor of dimension 2

In [ ]:
torch.tensor(features.shape) /

tensor([ 513.0000, 1875.5000])

In [26]:
print(test_dataset[0]['meta']['scenario_params'].keys())

dict_keys(['num_sources', 'signal_length', 'reverb_condition', 'microphone_array', 'noise_type', 'snr', 'activity_pattern', 'sources'])


In [27]:
[print(f"Scenario {i}: {test_dataset[i]['meta']['scenario_params']['microphone_array']}") for i in range(len(test_dataset))]

Scenario 0: BTE_left_IE_right
Scenario 1: BTE_rear
Scenario 2: BTE_front
Scenario 3: BTE
Scenario 4: BTE_left_IE_right
Scenario 5: BTE
Scenario 6: BTE_front_IE
Scenario 7: BTE_rear_IE
Scenario 8: BTE_front
Scenario 9: BTE
Scenario 10: BTE_right_IE_left
Scenario 11: BTE_rear_IE
Scenario 12: BTE_IE
Scenario 13: BTE_rear
Scenario 14: BTE_rear
Scenario 15: BTE_rear
Scenario 16: BTE_front
Scenario 17: IE
Scenario 18: BTE
Scenario 19: BTE_IE
Scenario 20: IE
Scenario 21: BTE_rear_IE
Scenario 22: BTE_rear
Scenario 23: BTE
Scenario 24: BTE_front
Scenario 25: BTE_IE
Scenario 26: BTE_left_IE_right
Scenario 27: BTE_left_IE_right
Scenario 28: BTE_left_IE_right
Scenario 29: BTE_front
Scenario 30: BTE_rear
Scenario 31: BTE_front_IE
Scenario 32: BTE_front_IE
Scenario 33: BTE
Scenario 34: IE
Scenario 35: BTE_rear
Scenario 36: BTE_right_IE_left
Scenario 37: IE
Scenario 38: BTE_left_IE_right
Scenario 39: BTE_right_IE_left
Scenario 40: IE
Scenario 41: IE
Scenario 42: IE
Scenario 43: BTE_rear
Scenario 44: 

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [28]:
print(scenario_data['meta']['scenario_params']["snr"])
[print(f"Scenario {i}: {test_dataset[i]['meta']['scenario_params']['snr']}") for i in range(len(test_dataset))]

14.541473909563834
Scenario 0: 14.918883991882074
Scenario 1: 6.685060761759645
Scenario 2: 9.368878528073077
Scenario 3: 10.36053389972547
Scenario 4: 11.630184529470526
Scenario 5: 9.778155568586875
Scenario 6: 14.570379480939282
Scenario 7: 11.695113667786181
Scenario 8: 7.976043881808385
Scenario 9: 14.96490488187237
Scenario 10: 13.774023534097108
Scenario 11: 8.979835620868055
Scenario 12: 7.433091623246437
Scenario 13: 5.755207895074662
Scenario 14: 13.29569528854609
Scenario 15: 13.898040761171519
Scenario 16: 9.75251929174975
Scenario 17: 13.466783112886771
Scenario 18: 13.794621732426862
Scenario 19: 6.5805971830133005
Scenario 20: 12.306183442967999
Scenario 21: 6.805063348541742
Scenario 22: 8.331274630855155
Scenario 23: 13.764525534344354
Scenario 24: 11.48507544176897
Scenario 25: 5.38840778923044
Scenario 26: 6.553515648462881
Scenario 27: 13.411421127550012
Scenario 28: 9.813676609554257
Scenario 29: 14.960736404895089
Scenario 30: 8.140342226655223
Scenario 31: 8.9461

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,

In [36]:
print(scenario_data["meta"]["scenario_params"]["activity_pattern"][0].keys())

dict_keys(['time', 'source', 'source_id'])


In [38]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from collections import defaultdict

# --- LaTeX/PGF Setup for Paper-Quality Figures ---
# Ensure this setup is consistent with your other plots
mpl.use("pgf")
mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",
    'font.family': 'serif',
    'text.usetex': True,
    'pgf.rcfonts': False,
})

# --- Configuration ---
# Figure Size
columnwidth_in = 244.6937 / 72
fig_width = columnwidth_in
fig_height = 1.4  # Adjust height as needed for clarity

# Font Sizes (in points)
title_fontsize = 9
label_fontsize = 9
tick_fontsize = 9

# --- Data Processing ---
# Extract the source activity event list
events = scenario_data["meta"]["scenario_params"]["activity_pattern"]
scenario_duration = 60.0  # Total duration of the scenario in seconds

# Get a sorted list of unique source IDs
source_ids = sorted(list(set(event['source_id'] for event in events)))

# Use a dictionary to track the activation time for each source
activation_times = {}
# Use a defaultdict to store the list of (start, duration) tuples for each source
activity_intervals = defaultdict(list)

for event in events:
    source_id = event['source_id']
    event_time = event['time']
    
    if event['type'] == 1:  # Activation
        # Store the activation time
        activation_times[source_id] = event_time
    elif event['type'] == -1 and source_id in activation_times:  # Deactivation
        # Calculate the duration and add the interval
        start_time = activation_times.pop(source_id)
        duration = event_time - start_time
        activity_intervals[source_id].append((start_time, duration))

# Handle sources that are still active at the end of the scenario
for source_id, start_time in activation_times.items():
    duration = scenario_duration - start_time
    activity_intervals[source_id].append((start_time, duration))

# --- Plotting ---
fig, ax = plt.subplots(figsize=(fig_width, fig_height), constrained_layout=True)

# Create the horizontal bar plot for each source
for i, source_id in enumerate(source_ids):
    intervals = activity_intervals.get(source_id, [])
    if intervals:
        ax.broken_barh(intervals, (i - 0.4, 0.8), facecolors='darkred')

# --- Configure Axes and Labels ---
ax.set_yticks(range(len(source_ids)))
# Use shortened labels for clarity if IDs are long
# short_labels = [f'Src {i+1} (...{sid[-8:]})' for i, sid in enumerate(source_ids)]
short_labels = [f'{i+1}' for i, _ in enumerate(source_ids)]

ax.set_yticklabels(short_labels, fontsize=tick_fontsize)

ax.set_xlabel("Time (s)", fontsize=label_fontsize)
ax.set_ylabel("Source ID", fontsize=label_fontsize)
# ax.set_title("Exemplary Source Activity Timeline", fontsize=title_fontsize)

ax.set_xlim(0, scenario_duration)
ax.set_ylim(-0.5, len(source_ids) - 0.5)
ax.tick_params(axis='both', labelsize=tick_fontsize)
ax.grid(axis='x', linestyle='--', alpha=0.6)

# Invert y-axis so Source 1 is at the top
ax.invert_yaxis()

# --- Add Source Count Overlay ---
# Create a secondary y-axis that shares the same x-axis
ax2 = ax.twinx()

# The gt_source_count and time_axis variables are from the previous cell.
# Plot the ground truth source count as a step plot.
ax2.step(time_axis, gt_source_count, where='post', color='cyan', linewidth=1.5, alpha=0.7)

# Configure the secondary y-axis
ax2.set_ylabel("\# Sources", color='black', fontsize=label_fontsize)
ax2.tick_params(axis='y', labelcolor='black', labelsize=tick_fontsize)
ax2.set_ylim(-0.625, gt_source_count.max() + 0.625)
ax2.set_yticks(np.arange(0, gt_source_count.max() + 1, 1)) # Integer ticks
ax2.grid(False)

# --- Save Figure ---
plt.savefig("source_activity_timeline_test.pdf", dpi=600)
plt.savefig("/data2/Henri/Journal3/framewiseSpeakerCounting/documents/ICASSP2026/figures/source_activity_timeline.pdf", dpi=600)

# To display in notebook, you may need to switch backends and show
# mpl.use("module://matplotlib_inline.backend_inline")
# plt.show()

<>:96: SyntaxWarning: invalid escape sequence '\#'
<>:96: SyntaxWarning: invalid escape sequence '\#'
/tmp/ipykernel_3076269/562855870.py:96: SyntaxWarning: invalid escape sequence '\#'
  ax2.set_ylabel("\# Sources", color='black', fontsize=label_fontsize)


KeyError: 'type'